# ReCTS detector control

> *"The clean experiment would be stock YOLOv11 plus your same fine-tuned OCR, so
> readers can see what your detector changes contribute to end-to-end 1-NED."*

Two detectors, one recogniser, identical everything else. Both arms are trained on
the ReCTS YOLO split with the published recipe (100 epochs, batch 64, imgsz 480,
cosine LR, `pretrained=True`) and read by the recogniser from notebook 1.

**Attach both inputs first:**
1. *Add Input -> Notebook Output* -> the recogniser notebook's output
2. *Add Input -> Datasets* -> `rishiksaisanthosh/yolo-obb-rects` (the published detector)

**Accelerator: GPU T4 x2. Runtime ~5.5 h** with `TRAIN_PAPER_ARM = False`
(4.4 h stock training, ~1 h submissions).

In [ ]:
SEED = 42
PROJECT = "/kaggle/working/runs/rects"

# The published detector already exists at seed 42, so the paper arm is reused rather
# than retrained. Set True to train it here instead - another 4.4 h per seed - which is
# what you need if you want the comparison at more than one seed.
TRAIN_PAPER_ARM = False

TASK3_CONFS = (0.3, 0.4, 0.5)
TASK4_CONFS = (0.4,)
DEVICE = "0,1"
REC_DEVICE = "cpu"     # see the install cell: paddle GPU breaks torch here

In [ ]:
!pip install -q gdown "ultralytics==8.3.189"
# CPU paddle on purpose. paddlepaddle-gpu installs its own nvidia-* wheels, which
# replace the NCCL that Kaggle's torch was built against, so `import ultralytics`
# then dies with "undefined symbol: ncclCommShrink". Recognition is the cheap half
# of this notebook; ~1 h on CPU beats fighting two CUDA stacks in one environment.
!pip install -q --timeout 180 --retries 10 paddlepaddle==3.2.0
!pip install -q paddleocr
!git clone -q --branch ablation-mscbam-probe https://github.com/SaiSanthosh1508/End-to-End-Text-Translation-Pipeline.git /kaggle/working/repo
import sys; sys.path.insert(0, "/kaggle/working/repo")

### Register the custom attention modules

`best.pt` pickles `CrossAttentionBlock` and `MultiScaleCBAM` under
`ultralytics.nn.modules.block`, so stock Ultralytics cannot unpickle it and the
paper arm's config cannot be built. `install_modules.py` patches a clean 8.3.189
install; it has to run before anything imports ultralytics, because patching
site-packages does not affect a module already loaded in this process.

In [ ]:
!cd /kaggle/working/repo && python ablation/install_modules.py

In [ ]:
import paddle
import torch
from ultralytics import YOLO
from paddleocr import TextRecognition

# install_modules.py puts the custom classes in ultralytics.nn.modules.custom, but
# best.pt pickles them under ...modules.block. Aliasing is what makes the checkpoint
# loadable; importing from block afterwards proves both halves worked.
from rects_control.detectors import register_pickle_aliases

register_pickle_aliases()
from ultralytics.nn.modules.block import CrossAttentionBlock, MultiScaleCBAM

print("torch", torch.__version__, "| CUDA:", torch.cuda.device_count(), "GPU(s)")
print("paddle", paddle.__version__, "| running recognition on", REC_DEVICE)
print("custom modules aliased:", CrossAttentionBlock.__name__, MultiScaleCBAM.__name__)
assert torch.cuda.device_count() >= 2, "need GPU T4 x2 for the detector arms"

## 1. Locate the recogniser and the published detector

Both come from attached inputs. Failing here costs nothing; failing after the
training run costs the session.

In [ ]:
from pathlib import Path

INPUT = Path("/kaggle/input")
roots = sorted(INPUT.glob("*")) if INPUT.exists() else []
print(f"{len(roots)} input(s):", [p.name for p in roots])
for root in roots:
    entries = sorted(root.rglob("*"))[:12]
    print(f"  {root.name}:", [str(e.relative_to(root)) for e in entries] or "EMPTY")
if not roots:
    raise RuntimeError("Nothing attached. Add the recogniser notebook output and yolo-obb-rects.")

# A large notebook output arrives as _output_.zip rather than extracted files, which is
# how the previous attempt found nothing. Pull just the recogniser out of it; random
# access means the 9 GB around it is never read.
import zipfile

STAGED = Path("/kaggle/working/staged_recognizer")
for archive in sorted(INPUT.glob("**/*.zip")):
    with zipfile.ZipFile(archive) as bundle:
        members = [m for m in bundle.namelist()
                   if "PP-OCRv5_rects_rec_infer/" in m and not m.endswith("/")]
        if members:
            print(f"extracting {len(members)} file(s) from {archive.name}")
            bundle.extractall(STAGED, members=members)
            break

SEARCH = ([STAGED] if STAGED.exists() else []) + roots

def locate(what, *patterns):
    for root in SEARCH:
        for pattern in patterns:
            hits = sorted(root.glob(pattern))
            if hits:
                return hits[0]
    seen = chr(10).join(f"    {p}" for root in SEARCH for p in sorted(root.rglob("*"))[:50])
    raise FileNotFoundError(f"{what} not found. Tried {patterns}, saw:{chr(10)}{seen}")

REC_DIR = locate("recogniser", "**/PP-OCRv5_rects_rec_infer/inference.yml",
                 "**/inference.yml").parent
PAPER_CKPT = locate("published ReCTS detector", "**/runs/obb/train3/weights/best.pt")
print("recogniser:", REC_DIR)
print("paper detector:", PAPER_CKPT)

## 2. ReCTS YOLO training split and the Task 3/4 test images

In [ ]:
!gdown -q 1wWgK4XvoBypaCprHcUFjEizhzPF4M35h -O /kaggle/working/rects_yolo.zip
!gdown -q 1mKqhPBDM-7BgUud69AYvQ7_BYmHqvFJC -O /kaggle/working/test1.zip
!gdown -q 1E8BlG5kh-JRAGOdYmCO75oi7Jy-UHHoW -O /kaggle/working/test2.zip
!unzip -q -o /kaggle/working/rects_yolo.zip -d /kaggle/working/rects_yolo
!unzip -q -o /kaggle/working/test1.zip -d /kaggle/working
!unzip -q -o /kaggle/working/test2.zip -d /kaggle/working

In [ ]:
roots = sorted({h.parent.parent for h in Path("/kaggle/working/rects_yolo").glob("**/images/train")
                if (h.parent.parent / "labels/train").is_dir()})
assert roots, "no YOLO tree in the ReCTS zip"
root = roots[0]

for split in ("train", "val"):
    n_img = len(list((root / f"images/{split}").glob("*")))
    n_lbl = len(list((root / f"labels/{split}").glob("*.txt")))
    print(f"{split}: {n_img} images, {n_lbl} labels")
    assert n_img and n_img == n_lbl, f"{split} split is empty or mismatched"

columns = {len(l.split()) for l in next((root / "labels/train").glob("*.txt")).read_text().split(chr(10)) if l.strip()}
assert columns == {9}, f"expected 9-column oriented labels, saw {columns}"

DATA_YAML = Path("/kaggle/working/rects.yaml")
DATA_YAML.write_text(f"train: {root}/images/train\nval: {root}/images/val\n\nnc: 1\n\nnames:\n  0: text\n")
print("data:", DATA_YAML.read_text())

In [ ]:
from rects_control.submission import image_files
n_test = len(image_files(Path("/kaggle/working")))
print(f"{n_test} Task 3/4 test images")
assert n_test > 1000, "test set looks truncated"

## 3. Smoke-test the whole chain before spending GPU hours

Detector to crop to recogniser to submission line, on one image, using the published
checkpoint that already exists. If PaddleOCR cannot load the exported model this is
where it surfaces - not 4.4 h from now, on a version that will save nothing.

In [ ]:
from PIL import Image
from ultralytics import YOLO
from rects_control.recognizer import PaddleRecognizer
from rects_control.submission import clockwise_points, crop_bgr, detect, image_files

recognizer = PaddleRecognizer(REC_DIR, device=REC_DEVICE)
probe = image_files(Path("/kaggle/working"))[0]
image = Image.open(probe)
quads = detect(YOLO(str(PAPER_CKPT)), image, 0.4)

crops = [c for c in (crop_bgr(image, q) for q in quads) if c is not None and c.size]
texts = recognizer(crops)
print(f"{probe.name}: {len(quads)} boxes, {sum(bool(t) for t in texts)} transcribed")
for quad, text in list(zip(quads, texts))[:5]:
    print("  ", ",".join(map(str, clockwise_points(quad, *image.size))), text)
assert any(texts), "recogniser returned nothing on a real test image"

## 4. Train the arms

`stock` is `a1_stock.yaml` - YOLOv11s-OBB as shipped. `paper` is `full_legacy.yaml`,
the deployed BiFPN + MS-CBAM + cross-attention model. Both get `nc=1`.

In [ ]:
from rects_control.detectors import adopt_published, train

if TRAIN_PAPER_ARM:
    paper = train("paper", DATA_YAML, Path(PROJECT), SEED, DEVICE)
else:
    paper = adopt_published(PAPER_CKPT, Path(PROJECT), SEED)

stock = train("stock", DATA_YAML, Path(PROJECT), SEED, DEVICE)

# Do not raise here: a failed version saves no output, which would discard the arm
# that succeeded along with the one that did not.
arms = {name: w for name, w in (("paper", paper), ("stock", stock)) if w}
for name in ("paper", "stock"):
    print(f"{name}: {arms.get(name, 'FAILED - see the traceback above')}")

## 5. Submissions

One `PaddleRecognizer` instance serves both arms, so any 1-NED difference is
attributable to detection. Task 3 is written at three confidences because the
detection operating point is a free parameter; Task 4 at the published 0.4.

In [ ]:
from rects_control.submission import write_task3, write_task4

out = Path("/kaggle/working/submissions"); out.mkdir(exist_ok=True)
summary = []

for arm, weights in arms.items():
    model = YOLO(str(weights))
    for conf in TASK3_CONFS:
        f = out / f"task3_{arm}_seed{SEED}_conf{conf:.1f}.txt"
        summary.append((arm, "task3", conf, write_task3(model, Path("/kaggle/working"), f, conf)))
        print(summary[-1])
    for conf in TASK4_CONFS:
        f = out / f"task4_{arm}_seed{SEED}_conf{conf:.1f}.txt"
        summary.append((arm, "task4", conf, write_task4(model, recognizer, Path("/kaggle/working"), f, conf)))
        print(summary[-1])

In [ ]:
import shutil
for f in sorted(out.glob("*.txt")):
    shutil.make_archive(str(f.with_suffix("")), "zip", f.parent, f.name)
    print(f.name, f.stat().st_size // 1024, "KB")

print("\nboxes written")
for arm, task, conf, n in summary:
    print(f"  {arm:6s} {task} conf={conf} -> {n}")

## 6. Upload

Save Version, download `/kaggle/working/submissions/*.zip`, and submit each to the
RRC ReCTS server — Task 3 for H-mean, Task 4 for 1-NED. The number the reviewer
asked for is `task4_stock` vs `task4_paper` 1-NED.

Report both, whichever way they fall. If the gap is small, the honest framing the
reviewer already offered is a deployment and systems contribution, which is still
publishable; the seed-to-seed spread in the MLT ablation table is the right yardstick
for deciding whether a given gap means anything.